1. Comparison of generated Relation Paths of an untrained llm model
-> compare the top-3 relation paths & to the ground truths

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.getenv("HF_TOKEN")
print()

In [4]:
! python src/qa_prediction/gen_rule_path_own.py \
        --model_name "Llama-2-7b-chat-hf" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --split validation \
        --output_path own-experiments/gen_rule_path

/home/noah/reasoning-on-graphs/.venv/lib/python3.10/site-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████| 2/2 [00:00<00:00, 199.50it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Save results to:  /home/noah/RoG-webqsp/Llama-2-7b-chat-hf/validation
  0%|                                                   | 0/246 [02:59<?, ?it/s]
Traceback (most recent call last):
  File "/home/noah/reasoning-on-graphs/src/qa_prediction/gen_rule_path_own.py", line 233, in <module>
    gen_path = gen_prediction(args)
  File "/home/noah/reasoning-on-graphs/src/qa_prediction/gen_rule_path_own.py", line 160, in gen_prediction
    raw_output = generate_seq(
  File "/home/noah/reasoning-on-graphs/src/qa_prediction/ge

2. Explainability: compare to newer models & also9 the factual truth
-> is the explanation always right?
-> is there an overreliance on the KG?

In [1]:
! python src/qa_prediction/predict_answer.py \
        --model_name "gemini-3.1-flash-lite" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path results/gen_rule_path/RoG-webqsp/RoG/validation/predictions_3_False.jsonl \
        --explain \
        --predict_path "own-experiments/KGQA/gemini-3.1-flash-lite-explain"


Load dataset from finished
Save results to:  own-experiments/KGQA/gemini-3.1-flash-lite-explain
Prepare pipline for inference...
 20%|████████▏                                 | 48/246 [03:53<15:47,  4.78s/it]Message: Based on the reasoning paths, please answer the given question. Please keep the answer as simple as possible and return all the possible answers as a list. Please explain your answer.

Reasoning Paths:
New York Mets -> sports.sports_team.championships -> 1986 World Series
New York Mets -> sports.sports_team.championships -> 1969 World Series

Question:
what year did the new york mets start?
Number of token: 237
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
100%|█████████████████████████████████████████| 246/246 [20:56<00:00,  5.11s/it]
Accuracy: 77.0440726165468 Hit: 87.39837398373983 F1: 37.54309829860147 Precision: 32.18738776373074 Recall: 77.0440726165468


In [9]:
! for model in kit.mistral-small-4-119b-a8b kit.minimax-m2.7-229b kit.gemma4-31b-it; do \
    python src/qa_prediction/predict_answer.py \
        --model_name "$model" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path results/gen_rule_path/RoG-webqsp/RoG/validation/predictions_3_False.jsonl \
        --explain \
        --predict_path "own-experiments/KGQA/$model-explain" ; \
    done

Load dataset from finished
Save results to:  own-experiments/KGQA/kit.mistral-small-4-119b-a8b-explain
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [32:14<00:00,  7.86s/it]
Accuracy: 72.40922834258406 Hit: 81.30081300813008 F1: 42.601773250121845 Precision: 36.16360184348488 Recall: 72.40922834258406
Load dataset from finished
Save results to:  own-experiments/KGQA/kit.minimax-m2.7-229b-explain
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [44:38<00:00, 10.89s/it]
Accuracy: 81.63034203455199 Hit: 90.65040650406505 F1: 18.30644002944172 Precision: 11.480640677865157 Recall: 81.63034203455199
Load dataset from finished
Save results to:  own-experiments/KGQA/kit.gemma4-31b-it-explain
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [34:25<00:00,  8.39s/it]
Accuracy: 71.84219096925449 Hit: 79.67479674796748 F1: 37.62655461684451 Precision: 29.791704306615852 Recall

## Plug-and-Play without system prompt or tools

In [4]:
! for model in gpt-oss-120b-wo-tools mistral-small-4-119b-a8b-wo-tools qwen35-397b-a17b-wo-tools minimax-m27-229b-wo-tools gemma4-31b-it-wo-tools; do \
    python src/qa_prediction/predict_answer.py \
        --model_name "$model" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path results/gen_rule_path/RoG-webqsp/RoG/validation/predictions_3_False.jsonl \
        --predict_path "own-experiments/KGQA/wo-tools/$model" ; \
    done

Load dataset from finished
Save results to:  own-experiments/KGQA/wo-tools/gpt-oss-120b-wo-tools
Prepare pipline for inference...
100%|████████████████████████████████████████| 246/246 [00:02<00:00, 111.79it/s]
Accuracy: 67.07821690827744 Hit: 78.04878048780488 F1: 74.20757742169138 Precision: 126.61971683935367 Recall: 67.07821690827744
Load dataset from finished
Save results to:  own-experiments/KGQA/wo-tools/mistral-small-4-119b-a8b-wo-tools
Prepare pipline for inference...
100%|████████████████████████████████████████| 246/246 [00:02<00:00, 119.29it/s]
Accuracy: 70.59888863605136 Hit: 82.92682926829268 F1: 65.7723165673358 Precision: 127.13114033304413 Recall: 70.59888863605136
Load dataset from finished
Save results to:  own-experiments/KGQA/wo-tools/qwen35-397b-a17b-wo-tools
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [20:00<00:00,  4.88s/it]
Accuracy: 73.48656801830319 Hit: 82.52032520325203 F1: 86.48760224119238 Precision: 160.756297

# retrieval-reasoning on few-shot generated predictions

In [9]:
! for model in kit.mistral-small-4-119b-a8b kit.minimax-m2.7-229b kit.gemma4-31b-it; do \
    python src/qa_prediction/predict_answer.py \
        --model_name "$model" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path own-experiments-few_shot/gen_rule_path/RoG-webqsp/google/gemma-2-9b/validation/predictions_3_False.jsonl \
        --predict_path "own-experiments-few_shot/KGQA/google-relation-$model" ; \
    done

Map: 100%|███████████████████████████| 246/246 [00:00<00:00, 1127.65 examples/s]
Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/google-relation-kit.mistral-small-4-119b-a8b
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [31:42<00:00,  7.73s/it]
Accuracy: 41.20003301127876 Hit: 59.34959349593496 F1: 29.15512366964039 Precision: 29.106754270645645 Recall: 41.20003301127876
Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/google-relation-kit.minimax-m2.7-229b
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [47:42<00:00, 11.64s/it]
Accuracy: 46.00010193544336 Hit: 63.41463414634146 F1: 11.016851405862505 Precision: 7.399932585877721 Recall: 46.00010193544336
Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/google-relation-kit.gemma4-31b-it
Prepare pipline for inference...
100%|███████████████████████████████████████| 246/246 [1:2

In [8]:
! for model in kit.mistral-small-4-119b-a8b kit.minimax-m2.7-229b kit.gemma4-31b-it; do \
    python src/qa_prediction/predict_answer.py \
        --model_name "$model" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path own-experiments-few_shot/gen_rule_path/RoG-webqsp/Llama-2-7b-chat-hf/validation/predictions_3_False.jsonl \
        --predict_path "own-experiments-few_shot/KGQA/llama-relation-$model" ; \
    done

Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/llama-relation-kit.mistral-small-4-119b-a8b
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [33:46<00:00,  8.24s/it]
Accuracy: 41.939085674851725 Hit: 58.94308943089431 F1: 29.856571084282297 Precision: 29.61126494272308 Recall: 41.939085674851725
Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/llama-relation-kit.minimax-m2.7-229b
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [48:01<00:00, 11.71s/it]
Accuracy: 47.37749102487772 Hit: 64.63414634146342 F1: 11.424219581877596 Precision: 7.31905678682077 Recall: 47.37749102487772
Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/llama-relation-kit.gemma4-31b-it
Prepare pipline for inference...
100%|███████████████████████████████████████| 246/246 [1:10:53<00:00, 17.29s/it]
Accuracy: 13.1288714672861 Hit: 17.073170731707318 F1: 10.56

In [1]:
! python src/qa_prediction/predict_answer.py \
        --model_name "gemini-3.1-flash-lite" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path own-experiments-few_shot/gen_rule_path/RoG-webqsp/Llama-2-7b-chat-hf/validation/predictions_3_False.jsonl \
        --predict_path "own-experiments-few_shot/KGQA/llama-relation-gemini-3.1-flash-lite" 

Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/llama-relation-gemini-3.1-flash-lite
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [14:00<00:00,  3.42s/it]
Accuracy: 46.18650946129726 Hit: 63.82113821138211 F1: 39.20310514873533 Precision: 42.32899189218198 Recall: 46.18650946129726


In [ ]:
! python src/qa_prediction/predict_answer.py \
        --model_name "gemini-3.1-flash-lite" \
        -d /home/noah/RoG-webqsp \
        --prompt_path prompts/general_prompt.txt \
        --add_rule \
        --split validation \
        --rule_path own-experiments-few_shot/gen_rule_path/RoG-webqsp/google/gemma-2-9b/validation/predictions_3_False.jsonl \
        --predict_path "own-experiments-few_shot/KGQA/google-relation-gemini-3.1-flash-lite"

Load dataset from finished
Save results to:  own-experiments-few_shot/KGQA/google-relation-gemini-3.1-flash-lite
Prepare pipline for inference...
100%|█████████████████████████████████████████| 246/246 [18:33<00:00,  4.53s/it]
Accuracy: 45.573873585503485 Hit: 62.60162601626016 F1: 38.55267503986572 Precision: 40.92068591339537 Recall: 45.573873585503485
/bin/bash: line 3: \: command not found


## evaluate results corercted

In [11]:
! python src/qa_prediction/evaluate_results_corrected.py \
    -d "own-experiments/KGQA/wo-tools/gpt-oss-120b-wo-tools/predictions.jsonl" \
    --top_k 1 \
    --cal_f1

Accuracy: 67.07821690827744 Hit: 69.91869918699187 F1: 74.20757742169138 Precision: 126.61971683935367 Recall: 67.07821690827744
